In [0]:
%run /Workspace/Users/azuser7248_mml.local@karthikirisoutlook.onmicrosoft.com/capstone-datasets/00_config

In [0]:
import requests

silver_config_url = "https://raw.githubusercontent.com/Santhoshpec/capstone-datasets/refs/heads/main/configs/silver_config.json"

silver_config = requests.get(
    silver_config_url
).json()

print(silver_config)

In [0]:
from pyspark.sql.functions import *

def create_silver_layer(config):

    entity = config["entity"]

    bronze_path = (
        f"{base_path}/bronze/{entity}/"
    )

    silver_path = (
        f"{base_path}/silver/{entity}/"
    )

    reject_path = (
        f"{base_path}/reject/{entity}/"
    )

    print(f"Processing : {entity}")

    # Read Bronze

    df = (
        spark.read
        .format("delta")
        .load(bronze_path)
    )

    # Datatype Conversion

    for column_name, data_type in config["casts"].items():

        if column_name in df.columns:

            df = (
                df.withColumn(
                    column_name,
                    col(column_name).cast(data_type)
                )
            )

    # Validation Condition

    validation_condition = (
        " AND ".join(
            config["rules"]
        )
    )

    # Valid Records

    silver_df = (
        df
        .filter(
            expr(validation_condition)
        )
        .withColumn(
            "_ProcessedTimestamp",
            current_timestamp()
        )
        .withColumn(
            "_IsRejected",
            lit(False)
        )
    )

    # Reject Records

    reject_df = (
        df
        .filter(
            ~expr(validation_condition)
        )
        .withColumn(
            "_ProcessedTimestamp",
            current_timestamp()
        )
        .withColumn(
            "_IsRejected",
            lit(True)
        )
    )

    # Write Silver

    (
        silver_df.write
        .format("delta")
        .mode("append")
        .save(silver_path)
    )

    # Write Reject

    (
        reject_df.write
        .format("delta")
        .mode("append")
        .save(reject_path)
    )

    print(
        f"Completed : {entity}"
    )

    print(
        f"Valid Records : {silver_df.count()}"
    )

    print(
        f"Rejected Records : {reject_df.count()}"
    )

In [0]:
for config in silver_config:

    create_silver_layer(config)

In [0]:
silver_customers = (
    spark.read
    .format("delta")
    .option(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        storage_account_key
    )
    .load(
        f"{base_path}/silver/customers/"
    )
)

display(silver_customers)

In [0]:
silver_customers = (
    spark.read
    .format("delta")
    .option(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        storage_account_key
    )
    .load(
        f"{base_path}/reject/customers/"
    )
)

display(silver_customers)